# Xem trên Drive đã lưu được gì

Notebook **chỉ đọc**. Không train, không xoá, không clone mã nguồn. Chạy mất
vài giây.

## Dùng khi nào

Phiên Colab bị ngắt giữa chừng và không biết đã chạy tới đâu.

`run_cv.py` nén kết quả sang Drive **sau mỗi fold**, và mỗi tệp nén chứa **toàn
bộ `runs/<thí nghiệm>/`** tại thời điểm đó — không phải riêng seed ấy. Nên tệp
mới nhất của một cấu hình là đủ để chạy tiếp.

Notebook này mở tệp nén ra xem *bên trong* mà không giải nén, rồi in ra fold nào
đã xong, fold nào còn thiếu, và lệnh cần chạy để tiếp tục.

## 1. Mount Drive

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## 2. Mọi tệp nén đang có

Cột cuối là giờ sửa. Tên chứa cả cấu hình lẫn seed.

In [2]:
!ls -la --time-style=+"%m-%d %H:%M" /content/drive/MyDrive/mobivital/*.zip \
  | awk '{printf "  %8.1f MB  %s %s  %s\n", $5/1048576, $6, $7, $8}' | sort -k4

       1.8 MB  09-07 00:07  /content/drive/MyDrive/mobivital/tn2_ds_tcn_c64_revin_mse_corr0.9_seed1.zip
      42.6 MB  09-06 00:43  /content/drive/MyDrive/mobivital/tn1_lstm_mse_corr0.9_seed2.zip
       5.3 MB  09-06 00:45  /content/drive/MyDrive/mobivital/tn1_ds_tcn_c64_mse_corr0.9_seed1.zip
       5.4 MB  09-04 00:57  /content/drive/MyDrive/mobivital/tn0.zip
       2.6 MB  09-07 01:19  /content/drive/MyDrive/mobivital/tn2_ds_tcn_c64_revin_mse_corr0.9_seed2.zip
       2.6 MB  09-07 01:19  /content/drive/MyDrive/mobivital/tn2_ds_tcn_revin.zip
       6.2 MB  09-06 01:55  /content/drive/MyDrive/mobivital/tn1_ds_tcn_c64_mse_corr0.9_seed2.zip
       9.3 MB  09-06 01:55  /content/drive/MyDrive/mobivital/tn1_tcn.zip
       0.8 MB  09-06 05:15  /content/drive/MyDrive/mobivital/tn1_lstm_h67_mse_corr0.9_seed0.zip
       1.7 MB  09-06 05:55  /content/drive/MyDrive/mobivital/tn1_lstm_h67_mse_corr0.9_seed1.zip
       0.0 MB  09-07 06:03  /content/drive/MyDrive/mobivital/tn1_mix_linear_p10_lpf5_mse

## 3. Chọn cấu hình muốn xem

Đổi `LOC` thành phần tên đặc trưng của cấu hình. Ví dụ `c192`, `c64`, `gru`,
`mix_linear`. Để trống thì xem tất cả.

In [ ]:
LOC = "c192"

## 4. Fold nào đã xong

Đọc `summary.csv` **bên trong** tệp nén, không giải nén ra đĩa.

In [4]:
import glob, subprocess, csv, io, os
z = sorted(glob.glob("/content/drive/MyDrive/mobivital/*%s*.zip" % LOC), key=os.path.getmtime)
print("%d tệp khớp '%s'. Mới nhất: %s\n" % (len(z), LOC, os.path.basename(z[-1]) if z else "—"))
for f in z: print("  %s   %s" % (os.path.basename(f)[:70],
    subprocess.run(["date","-r",f,"+%m-%d %H:%M"],capture_output=True,text=True).stdout.strip()))

3 tệp khớp 'c192'. Mới nhất: tn1_ds_tcn_c192_k3_n4_none_do0.2_dpel_mse_corr0.9_seed0.zip

  tn_test_ds_tcn_c192.zip   09-07 08:00
  tn_test_ds_tcn_c192_k5_n4_do0.2_mse_corr0.9_seed0.zip   09-07 08:00
  tn1_ds_tcn_c192_k3_n4_none_do0.2_dpel_mse_corr0.9_seed0.zip   09-07 15:00


In [5]:
ten = subprocess.run(["unzip","-Z1",z[-1]],capture_output=True,text=True).stdout
csv_path = [x for x in ten.split() if x.endswith("summary.csv")][0]
raw = subprocess.run(["unzip","-p",z[-1],csv_path],capture_output=True,text=True).stdout
rows = [r for r in csv.DictReader(io.StringIO(raw)) if LOC in r["run_id"]]
print("%s  ->  %d dòng khớp '%s'" % (csv_path, len(rows), LOC))

tn1/summary.csv  ->  4 dòng khớp 'c192'


## 5. Bảng fold đã xong

Mỗi cấu hình cần **4 fold × 3 seed = 12 dòng**, cộng dòng `TONG` mỗi seed nếu
seed đó xong đủ 4 fold.

In [ ]:
import re
from collections import defaultdict
FOLD = ["val_AB", "val_CE", "val_DF", "val_KL"]
xong = defaultdict(set)

In [ ]:
for r in rows:
    m = re.match(r"(.+?)_seed(\d+)(?:_(val_\w+)|_tong)?$", r["run_id"])
    if m: xong[(m.group(1), int(m.group(2)))].add(m.group(3) or "TONG")
for cfg, s in sorted(xong):
    co = xong[(cfg, s)]
    print("  seed %d  %s  %s" % (s, " ".join("%-7s" % (f if f in co else "·") for f in FOLD),
          "xong" if len(co & set(FOLD)) == 4 else "còn %d fold"
          % (4 - len(co & set(FOLD)))))

## 6. Điểm từng fold đã có

In [8]:
for r in sorted(rows, key=lambda r: r["run_id"]):
    print("  %-62s %s" % (r["run_id"][:62], r.get("score_macro","")))

  ds_tcn_c192_k3_n4_none_do0.2_dpel_mse_corr0.9_seed0_val_AB     0.8013471237187482
  ds_tcn_c192_k3_n4_none_do0.2_dpel_mse_corr0.9_seed0_val_CE     0.7809160498023013
  ds_tcn_c192_k3_n4_none_do0.2_dpel_mse_corr0.9_seed0_val_DF     0.6266600344428807
  ds_tcn_c192_k3_n4_none_do0.2_dpel_mse_corr0.9_seed0_val_KL     0.8419339795087752


## 7. Lệnh để chạy tiếp

Chép hai dòng dưới vào notebook train, **chèn ngay sau ô khôi phục dữ liệu và
trước các ô train**. Ô clone xoá `runs/` nên không có bước này thì `run_cv.py`
train lại từ đầu.

In [9]:
print("!unzip -oq /content/drive/MyDrive/mobivital/%s -d runs/" % os.path.basename(z[-1]))
print("!ls runs/*/ | grep %s | head" % LOC)

!unzip -oq /content/drive/MyDrive/mobivital/tn1_ds_tcn_c192_k3_n4_none_do0.2_dpel_mse_corr0.9_seed0.zip -d runs/
!ls runs/*/ | grep c192 | head


Sau đó bấm các ô train như thường. Với mỗi fold đã có kết quả, `run_cv.py`
sẽ in

```
đã có kết quả 0.xxxx — bỏ qua, không train lại
```

và chỉ train phần còn thiếu.